## 局部时深表（Local Time-Depth Table）

这个 notebook 演示一条最小可运行链路：

- 显式指定一口井的 LAS 路径
- 从井头文件读取 `Well datum value` 作为 `kb`
- 把局部井段的 MD 重新基准到 0（局部参考系）
- 基于 Vp 构建局部 `TimeDepthTable`
- 使用 `plot_td_table` 可视化


In [ ]:
import sys
from pathlib import Path

import lasio
import matplotlib.pyplot as plt
import numpy as np

# 从 notebooks/ 回到仓库根目录
repo_root = Path.cwd().resolve().parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from cup.petrel.load import extract_vp_log_from_las, import_well_heads_petrel
from cup.utils.process import recompute_trace_basis
from wtie.processing import grid
from wtie.utils import viz


In [ ]:
# 1) 显式指定数据路径（可按需替换井名）
well_name = "2-ANP-2A-RJS"
las_path = repo_root / "data" / "vertical_well_las_target" / f"{well_name}.Las"
well_heads_path = repo_root / "data" / "raw" / "well_heads"

assert las_path.exists(), f"LAS 不存在: {las_path}"
assert well_heads_path.exists(), f"well_heads 不存在: {well_heads_path}"

# 2) 读取 Vp
las_file = lasio.read(las_path)
vp_md = extract_vp_log_from_las(las_file)

# 若有多个 Vp 候选曲线，可改为显式指定 mnemonic：
# vp_md = extract_vp_log_from_las(las_file, curve_mnemonic="DTC")

# 3) 从井头读取 kb（Well datum value）
well_heads_df = import_well_heads_petrel(well_heads_path)
row = well_heads_df.loc[well_heads_df["Name"] == well_name]
assert len(row) == 1, f"在 well_heads 中未唯一匹配井名: {well_name}"
kb = float(row.iloc[0]["Well datum value"])

print(f"well_name = {well_name}")
print(f"las_path = {las_path}")
print(f"kb = {kb:.3f} m")
print(f"Vp samples = {vp_md.size}, md range = [{vp_md.basis[0]:.3f}, {vp_md.basis[-1]:.3f}] m")

In [ ]:
# 4) 读取井分层，并按 BVE_TOP 到 ITP_BOT 选择井段
from cup.petrel.load import import_well_tops_petrel

well_tops_path = repo_root / "data" / "raw" / "well_tops"
assert well_tops_path.exists(), f"well_tops 不存在: {well_tops_path}"

well_tops_df = import_well_tops_petrel(well_tops_path)
tops_well = well_tops_df.loc[well_tops_df["Well"] == well_name].copy()

top_surface_name = "BVE_TOP"
base_surface_name = "ITP_BOT"

top_rows = tops_well.loc[tops_well["Surface"] == top_surface_name]
base_rows = tops_well.loc[tops_well["Surface"] == base_surface_name]

assert len(top_rows) > 0, f"井 {well_name} 缺少层位: {top_surface_name}"
assert len(base_rows) > 0, f"井 {well_name} 缺少层位: {base_surface_name}"

md_top = float(top_rows["MD"].min())
md_bot = float(base_rows["MD"].max())
assert md_bot > md_top, f"层位区间非法: {top_surface_name}={md_top}, {base_surface_name}={md_bot}"

# 与 Vp 实际覆盖范围求交，避免超界
md0 = max(float(vp_md.basis[0]), md_top)
md1 = min(float(vp_md.basis[-1]), md_bot)
assert md1 > md0, "层位区间与 Vp 井段无交集"

mask = (vp_md.basis >= md0) & (vp_md.basis <= md1)
assert mask.sum() > 10, "层位井段采样点太少"

vp_local = grid.Log(
    values=vp_md.values[mask],
    basis=vp_md.basis[mask],
    basis_type="md",
    name=vp_md.name,
    unit=vp_md.unit,
)

print(f"tops interval: {top_surface_name} -> {base_surface_name}")
print(f"tops md interval = [{md_top:.3f}, {md_bot:.3f}] m")
print(f"used md interval = [{md0:.3f}, {md1:.3f}] m, n={vp_local.size}")

# 5) 把局部井段重置到局部 MD=0 参考系
vp_local0 = recompute_trace_basis(vp_local, basis_start=0.0)

# 对应直井近似下的 WellPath（tvdss 不传时会自动用 md-kb）
wp_local = grid.WellPath(md=vp_local0.basis, kb=kb)

# 6) 直接由 Vp 积分得到局部时深表，顶部定义为 t=0
local_tdt = grid.TimeDepthTable.get_tvdss_twt_relation_from_vp(
    vp_local0,  # type: ignore
    wp=wp_local,
    origin=0.0,
)

print(f"local_tdt size = {local_tdt.size}")
print(f"twt range = [{local_tdt.twt[0]:.4f}, {local_tdt.twt[-1]:.4f}] s")
print(f"tvdss range = [{local_tdt.tvdss[0]:.3f}, {local_tdt.tvdss[-1]:.3f}] m")

# 8) 可视化
fig, ax = viz.plot_td_table(local_tdt, plot_params={"label": "local tdt", "linewidth": 1.8})
ax.legend(loc="best")
plt.show()